## Simulation

In [ ]:
from nanover.openmm import OpenMMSimulation
from nanover.mdanalysis.converter import frame_data_to_mdanalysis
from nanover.trajectory import FrameData

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

## iMD runner + client

In [ ]:
from nanover.app import OmniRunner
from nanover.websocket import NanoverImdClient

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="MOVEABLE RESTRAINTS")
imd_runner.load(0)
imd = imd_runner.app_server.imd

client = NanoverImdClient.from_runner(imd_runner)

In [ ]:
with client.root_selection.modify() as selection:
    selection.renderer = "cartoon"

## Utilties

In [ ]:
from nanover.jupyter import NanoverJupyterUtilities, Mode

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.use_recording_commands()
utilities.use_interaction_modes()

## Restraints

In [ ]:
import numpy as np
from nanover.app import RenderingSelection
from nanover.imd.imd_state import INTERACTION_PREFIX, ParticleInteraction

RESTRAINT_PREFIX = f"{INTERACTION_PREFIX}MOVEABLE-RESTRAINT."

NEXT_RESTRAINT_INDEX = 0
MOVEABLE_RESTRAINTS: dict[str, ParticleInteraction] = {}
SELECTIONS: dict[str, RenderingSelection] = {}
ACTIVE_RESTRAINTS: set[str] = set()

RENDERER = {
    "render": {
        "particle.scale": .1,
        "bond.scale": 0.0,
        "type": "ball and stick"
    },
    "color": "CornflowerBlue",
}

def add_restraint(indexes):
    global NEXT_RESTRAINT_INDEX
    indexes = [int(index) for index in indexes]

    key = f"{RESTRAINT_PREFIX}{NEXT_RESTRAINT_INDEX}"
    NEXT_RESTRAINT_INDEX += 1
    restraint = ParticleInteraction((0, 0, 0), indexes)
    restraint.scale = 1000
    restraint.interaction_type = "spring"
    MOVEABLE_RESTRAINTS[key] = restraint

    utilities.notify_all("ADD RESTRAINT")

    with client.create_selection(key, indexes).modify() as selection:
        selection.renderer = RENDERER
        selection.velocity_reset = True
        selection.interaction_method = "group"
        SELECTIONS[key] = selection

    enable_restraint(key)

    return key


def remove_restraint(key):
    disable_restraint(key)
    client.remove_selection(SELECTIONS[key])
    MOVEABLE_RESTRAINTS.pop(key, None)
    utilities.objects.remove_shape(key)
    utilities.objects.remove_line(key)


def enable_restraint(key):
    if key in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.add(key)
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    restraint = MOVEABLE_RESTRAINTS[key]
    restraint.position = np.mean(positions[restraint.particles], axis=0)
    utilities.notify_all("ENABLE RESTRAINT")
    imd.insert_interaction(key, restraint)


def disable_restraint(key):
    if key not in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.remove(key)
    imd.remove_interaction(key)

## Visuals

In [ ]:
import numpy as np
from nanover.jupyter import FrameListener

# TODO: why does unsanitised state update break the frame stream but only for this thread?

class MobileRestraintVisuals(FrameListener):
    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        with utilities.objects as objects:
            try:
                for key, restraint in MOVEABLE_RESTRAINTS.items():
                    centroid = np.mean(full_frame.particle_positions[restraint.particles], axis=0)
                    distance = np.linalg.norm(centroid - restraint.position)
                    strain = min(distance / .5, 1.0)
                    color = [1.0, 1.0 - strain, 1.0 - strain, 1.0]
                    objects.update_shape(key, position=restraint.position, scale=.5, color=color)
                    objects.update_line(key, positions=[restraint.position, centroid], scale=.1, color=color)
            except Exception as e:
                objects.update_label("error", text=str(e), position=[0.0, 0.0, 0.0], size=.2)


visuals = MobileRestraintVisuals.from_runner(imd_runner)
visuals.start()

## Commands

In [ ]:
def clear_restraints():
    for restraint in list(MOVEABLE_RESTRAINTS.keys()):
        remove_restraint(restraint)
    MOVEABLE_RESTRAINTS.clear()

imd_runner.app_server.register_command("user/restraints/clear", clear_restraints)

In [ ]:
class InteractMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        if key.startswith(RESTRAINT_PREFIX):
            return

        interacted_indexes = set(interaction.particles)
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                disable_restraint(key)

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        if key.startswith(RESTRAINT_PREFIX):
            return

        interacted_indexes = set(interaction.particles)
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                enable_restraint(key)

utilities.add_interaction_mode(InteractMode, "move")

In [ ]:
class ToggleMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        if key.startswith(RESTRAINT_PREFIX):
            return

        interacted_indexes = set(interaction.particles)
        removed = False
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                remove_restraint(key)
                removed = True
        if not removed:
            add_restraint(interaction.particles)

utilities.add_interaction_mode(ToggleMode, "toggle")

In [ ]:
isinstance(np.array([1.0])[0], np.number), isinstance(1.0, np.number), isinstance(np.array([1.0]).tolist()[0], float)